# Traceability notebook: alpha vs pull_increase correlation

Every number reported in `hb_model_findings_summary.md` (Finding 2) is reproduced
here from raw inputs, nothing is hardcoded. Inputs needed:

1. `data01_direction4priors.csv` (the raw behavioral data, same file used throughout
   this project)
2. `alpha_by_subject.csv` (each subject's fitted `alpha` from the HB model, with the
   exact checkpoint file it came from, for provenance)

Everything downstream, `pull_increase`, the merge, the correlations, the robustness
checks, is computed in this notebook, not pasted in from a prior conversation.


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

RAW_CSV_PATH = "data01_direction4priors.csv"          # adjust path if needed
ALPHA_CSV_PATH = "alpha_by_subject.csv"                 # adjust path if needed

raw = pd.read_csv(RAW_CSV_PATH)
alpha_df = pd.read_csv(ALPHA_CSV_PATH)
print(f"raw: {raw.shape[0]:,} rows, {raw.shape[1]} columns")
print(alpha_df)


## Step 1: rebuild `signed_prior_pull` from raw trial data

`signed_prior_pull` = the part of a subject's response error that points toward
the fixed prior mean (225 degrees), rather than away from it. Positive means the
response leaned toward the prior relative to the true motion direction shown.

Trials where the prior mean and the true motion direction are within 10 degrees
of each other are excluded, "pull toward the prior" isn't meaningfully separable
from "pull toward the true direction" when the two are almost the same place.


In [ ]:
data = raw.copy()
data['estimate_deg'] = np.degrees(np.arctan2(data.estimate_y, data.estimate_x)) % 360
data.loc[data.estimate_deg == 0, 'estimate_deg'] = 360
data = data.dropna(subset=['estimate_deg', 'motion_direction']).copy()
data['error_deg'] = ((data.estimate_deg - data.motion_direction + 180) % 360) - 180
data['block_id'] = data['subject_id'].astype(str) + '_' + data['run_id'].astype(str)

def wrap_signed_deg(a):
    return ((np.asarray(a) + 180) % 360) - 180

data['motion_offset_from_prior'] = wrap_signed_deg(data.motion_direction - data.prior_mean)
direction_to_prior = np.sign(-data.motion_offset_from_prior)
data['signed_prior_pull'] = direction_to_prior * data.error_deg

pull_data = data[data.motion_offset_from_prior.abs() >= 10].copy()
print(f"{len(data) - len(pull_data)} trials excluded (prior and motion direction within 10 degrees)")
print(f"{len(pull_data)} trials remain")


## Step 2: early vs. late trials within a block, per subject and coherence

"Early" = the first 15 trials of a block. "Late" = trial 101 onward. `pull_increase`
= mean late `signed_prior_pull` minus mean early `signed_prior_pull`. Positive means
leaning on the prior *more* by the end of a block.


In [ ]:
pull_data = pull_data.sort_values(['subject_id', 'run_id', 'trial_index'])
pull_data['trials_into_block'] = pull_data.groupby('block_id').cumcount() + 1

early = pull_data[pull_data.trials_into_block.between(1, 15)]
late  = pull_data[pull_data.trials_into_block >= 101]

def summarize(df, label):
    return df.groupby(['subject_id', 'motion_coherence'])['signed_prior_pull'].mean().rename(label)

early_s = summarize(early, 'early_pull')
late_s  = summarize(late,  'late_pull')

combined = pd.concat([early_s, late_s], axis=1).dropna()
combined['pull_increase'] = combined['late_pull'] - combined['early_pull']
combined = combined.reset_index()

pivot = combined.pivot(index='subject_id', columns='motion_coherence', values='pull_increase')
pivot.columns = [f'pull_increase_{c}' for c in pivot.columns]
pivot['pull_increase_avg'] = pivot.mean(axis=1)
pivot = pivot.reset_index().rename(columns={'subject_id': 'subject'})
print(pivot.round(3))


## Step 3: merge with alpha and compute correlations

In [ ]:
merged = alpha_df.merge(pivot, on='subject')
print(merged.round(4))
print()

results = []
for col in ['pull_increase_avg', 'pull_increase_0.06', 'pull_increase_0.12', 'pull_increase_0.24']:
    r, p = stats.pearsonr(merged['alpha'], merged[col])
    results.append((col, r, p))
    print(f"alpha vs {col}: r={r:.3f}, p={p:.3f}")

merged.to_csv('alpha_pull_merged_traceable.csv', index=False)


## Step 4: robustness checks on the one significant result (6% coherence)

- Drop the most extreme subject and recheck
- Spearman (rank-based) correlation, less sensitive to any single outlier


In [ ]:
most_extreme_subject = merged.loc[merged['pull_increase_0.06'].abs().idxmax(), 'subject']
print(f"Most extreme subject on pull_increase_0.06: subject {most_extreme_subject}")

no_outlier = merged[merged.subject != most_extreme_subject]
r_no_outlier, p_no_outlier = stats.pearsonr(no_outlier['alpha'], no_outlier['pull_increase_0.06'])
print(f"Excluding subject {most_extreme_subject}: r={r_no_outlier:.3f}, p={p_no_outlier:.3f}, n={len(no_outlier)}")

r_spearman, p_spearman = stats.spearmanr(merged['alpha'], merged['pull_increase_0.06'])
print(f"Spearman (all 12 subjects): r={r_spearman:.3f}, p={p_spearman:.3f}")


## Step 5: multiple-comparisons check

4 correlations were tested (avg + 3 coherence levels). A Bonferroni-corrected
threshold for calling any one of them "significant" at an overall 0.05 level would
require p < 0.0125 for that individual test.


In [ ]:
alpha_level = 0.05
n_tests = 4
bonferroni_threshold = alpha_level / n_tests
print(f"Bonferroni-corrected threshold: p < {bonferroni_threshold:.4f}")
for col, r, p in results:
    survives = "survives" if p < bonferroni_threshold else "does NOT survive"
    print(f"  {col}: p={p:.4f} -> {survives} Bonferroni correction")
